# LAB006: Build agentic knowledge bases — next-level RAG with Azure AI Search

This lab demonstrates Azure AI Search's agentic retrieval capabilities through 5 hands-on notebook exercises. You'll build a Knowledge Base that connects chat models to multiple data sources, extend it with MCP (Model Context Protocol) knowledge sources, consume it as an MCP endpoint, and connect it to a Foundry Agent.

<details>
<summary><b>🤖 What is agentic RAG?</b></summary>

Unlike traditional RAG, agentic retrieval analyzes conversation context to understand user intent, decomposes complex queries into focused subqueries, and executes searches concurrently across multiple sources. It ranks and filters results using semantic understanding, then synthesizes answers with citations back to source documents.
</details>

<details>
<summary><b>🧠 What is a knowledge base?</b></summary>

In Azure AI Search, a **knowledge base** is a top-level resource that connects a chat completion model to one or more **knowledge sources** (searchable indexes or remote data sources). It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized. Once created, you can update it at any time.
</details>

## Lab notebooks

- **Part 1: Build a multi-source knowledge base** (this notebook) — Create indexed knowledge sources, build a knowledge base with retrieval/answer instructions, and query with citation-backed answers.
- **Part 2: Consume KB as an MCP endpoint** — Configure the knowledge base as an MCP server for GitHub Copilot CLI.
- **Part 3: Add a non-authenticated MCP source (Learn)** — Connect Microsoft Learn's MCP server as a live knowledge source.
- **Part 4: Add an authenticated MCP source (GitHub)** — Connect GitHub's MCP server with Bearer token authentication.
- **Part 5: Connect KB to a Foundry agent** — Wire the knowledge base into a Microsoft Foundry Agent for conversational grounding.

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

- **Azure AI Search**
  - `hrdocs` index: HR policies, handbook, company info
  - `healthdocs` index: health benefits, insurance, coverage
  - Semantic ranking enabled, pre-computed embeddings

- **Azure OpenAI**
  - `gpt-4.1`: Chat completion for reasoning and synthesis
  - `text-embedding-3-large`: Embedding model for vector search

All documents are already indexed and vectorized. You'll focus on the agentic retrieval APIs, not data preparation.

> **📌 Tips**
>
> - Start with Part 1 to understand core concepts, then continue through Parts 2–5.
>
> - Each notebook is self-contained — you can run them independently or in sequence.
>
> - ‼️ **Read the outputs**. Activity logs show how agentic retrieval works under the hood.

Let's start with Part 1! 👇

# Part 1: Build a multi-source knowledge base

In this notebook, you'll create two knowledge sources (HR docs and health docs), combine them into a single knowledge base with Azure OpenAI, and query using agentic retrieval. You'll also learn how to control source selection and answer formatting with natural language instructions.

## Step 1: Load environment variables

Load the configuration for your Azure resources. The `.env` file in the workspace root has everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

> **ℹ️ Note**
>
> - The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
>
> - You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

> **⚠️ Troubleshooting**
>
> If code cells get stuck and keep spinning, select **Restart** from the notebook toolbar at the top. If the issue persists after a couple of tries, close VS Code completely and reopen it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

# Azure AI Search configuration
endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
credential = AzureKeyCredential(os.environ["AZURE_SEARCH_ADMIN_KEY"])

# Knowledge base name
knowledge_base_name = "hr-and-health-docs-knowledge-base"

# Azure OpenAI configuration
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
azure_openai_chatgpt_deployment = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
azure_openai_chatgpt_model_name = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

print("Environment variables loaded")

## Step 2: Create two knowledge sources

A **knowledge source** connects your knowledge base to actual data. You'll create two knowledge sources pointing to different indexes:

- **healthdocs-knowledge-source**: Points to the `healthdocs` index (334 document chunks about health benefits and insurance)
- **hrdocs-knowledge-source**: Points to the `hrdocs` index (50 document chunks about HR policies)

Both use the same field configuration:
- `blob_path`: Where the original document lives (used for citation links)
- `snippet`: The actual text content

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

ks = SearchIndexKnowledgeSource(
    name="healthdocs-knowledge-source",
    description="Zava health documents: health benefits and insurance plan information",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name="healthdocs",
        source_data_fields=[
            SearchIndexFieldReference(name="blob_path"),
            SearchIndexFieldReference(name="snippet"),
        ],
        search_fields=[SearchIndexFieldReference(name="snippet")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=ks)
print(f"Knowledge source '{ks.name}' created or updated successfully.")

ks = SearchIndexKnowledgeSource(
    name="hrdocs-knowledge-source",
    description="Zava HR documents: company policies, job roles, workplace guidelines, and general benefits",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name="hrdocs",
        source_data_fields=[
            SearchIndexFieldReference(name="blob_path"),
            SearchIndexFieldReference(name="snippet"),
        ],
        search_fields=[SearchIndexFieldReference(name="snippet")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=ks)
print(f"Knowledge source '{ks.name}' created or updated successfully.")

## Step 3: Create combined knowledge base

A **knowledge base** is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An AI model (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

The `output_mode=ANSWER_SYNTHESIS` tells the knowledge base to use the AI model to synthesize natural language answers from the retrieved documents. The LLM decides what information is relevant and how to present it.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeRetrievalOutputMode,
    KnowledgeSourceReference,
)

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=azure_openai_endpoint,
    deployment_name=azure_openai_chatgpt_deployment,
    model_name=azure_openai_chatgpt_model_name,
    api_key=azure_openai_key,
)

knowledge_base = KnowledgeBase(
    name=knowledge_base_name,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name="healthdocs-knowledge-source"),
        KnowledgeSourceReference(name="hrdocs-knowledge-source"),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base_name}' created or updated successfully.")

## Step 4: Query multiple sources

Let's ask a two-part question that spans both knowledge sources:
- "What is the responsibility of the Zava CEO?" (HR-related)
- "What health plan would you recommend for the best mental health coverage?" (Health-related)

The knowledge base uses agentic retrieval to:
1. Analyze the query to understand two different topics
2. Decompose the query into focused subqueries
3. Determine which knowledge sources are relevant for each subquery
4. Run searches concurrently against the selected sources
5. Synthesize a coherent answer from both sources

In [ ]:
import re

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=endpoint, knowledge_base_name=knowledge_base_name, credential=credential
)

healthdocs_ks_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name="healthdocs-knowledge-source",
    include_references=True,
    include_reference_source_data=True,
)
hrdocs_ks_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name="hrdocs-knowledge-source",
    include_references=True,
    include_reference_source_data=True,
)

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="What is the responsibility of the Zava CEO? "
                    "What health plan would you recommend if they wanted the best coverage for mental health services?"
                )
            ],
        )
    ],
    knowledge_source_params=[healthdocs_ks_params, hrdocs_ks_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

## Step 5: Map citations to knowledge sources

This helper function traces each citation in the answer back to its original knowledge source. It shows you whether information came from `hrdocs` or `healthdocs`, so you can verify the knowledge base is routing queries to the right sources.

In [ ]:
def find_source_of_reference(result, reference_id):
    activity_id = None
    for reference in result.references:
        if reference.id == reference_id:
            activity_id = reference.activity_source
            break
    for activity in result.activity:
        if activity.id == activity_id:
            return activity.knowledge_source_name
    return None


def cite_sources(result):
    references = re.findall(r"\[ref_id:(\d+)\]", result.response[0].content[0].text)
    sources = {}
    for ref_id in references:
        source_info = find_source_of_reference(result, ref_id)
        if source_info:
            sources[ref_id] = source_info
    return sources


sources = cite_sources(result)
print("Cited sources:", sources)

## Step 6: Review activity log

The activity log shows the detailed trace of the agentic retrieval process — what subqueries were generated, which sources were searched, and how results were ranked. This is essential for debugging and understanding the agent's reasoning.

In [ ]:
import json

import pandas as pd

activity_types = [{"type": a.type} for a in result.activity]
df = pd.DataFrame(activity_types)
print("Activity Log Steps")
display(df)

print("\nActivity Details")
print(json.dumps([a.as_dict() for a in result.activity], indent=2))

## Step 7: Guide source selection with retrieval instructions

**Retrieval instructions** let you guide the knowledge base's source selection using natural language. Instead of forcing it to query everything, you provide hints like "Use healthdocs for health questions, hrdocs for HR questions."

The knowledge base still makes the final decision, but now with your guidance — it can skip irrelevant sources for better efficiency.

In [ ]:
knowledge_base.retrieval_instructions = (
    "If the question is about health benefits or insurance specifically, use healthdocs. "
    "Otherwise, use hrdocs for other HR-related questions."
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base_name}' updated with retrieval instructions.")

In [ ]:
req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text="What health benefits are there?")],
        )
    ],
    knowledge_source_params=[healthdocs_ks_params, hrdocs_ks_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

sources_queried = [
    activity.knowledge_source_name
    for activity in result.activity
    if activity.type == "searchIndex"
]
print("Queried sources:", sources_queried)

## Step 8: Format answers with answer instructions

**Answer instructions** control how the knowledge base formats its response. They don't change which sources are queried or what information is retrieved — just how it's presented.

In [ ]:
knowledge_base.answer_instructions = (
    "Always use a Markdown list format when providing answers. Each list item should be on a separate line."
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base_name}' updated with answer instructions.")

In [ ]:
result = knowledge_base_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

## Summary

You've completed Part 1! You created two indexed knowledge sources, built a multi-source knowledge base with Azure OpenAI, and queried using agentic retrieval with citation-backed answers.

**Key concepts:**
- A single knowledge base can reference multiple knowledge sources
- The knowledge base intelligently selects which sources to query based on the question
- `retrieval_instructions` guide source selection with natural language
- `answer_instructions` control response formatting without changing content
- Activity logs show how the agent processed your query

➡️ Continue to [Part 2: Consume KB as an MCP Endpoint](part2-kb-as-mcp-endpoint.ipynb) to learn how to use this knowledge base as an MCP server for developer tools like GitHub Copilot CLI.